# Variant clusters

Qualifying disease credible sets are grouped into likely independent causal signals:
connected components over significant colocalisations and shared lead variants. The number
of distinct diseases in a cluster is its variant pleiotropy score (vPS). Methods
"Variant-level pleiotropy modelling".

Disease identity uses the ontology-resolved `diseaseIds` column, as everywhere else in the
pipeline. The raw curator column `traitFromSourceMappedIds` is reported alongside for
comparison: 26 of its ids no longer exist in the release ontology.

Writes `variant_clusters`, `cluster_membership`.

In [1]:
import pandas as pd

from manuscript_methods import clusters, paper

In [2]:
cs = clusters.load_credible_sets()
print("qualifying disease credible sets:", len(cs))

edges = clusters.load_edges(set(cs["studyLocusId"]))
print("colocalisation edges within the set:", len(edges))

components = clusters.cluster(list(zip(cs["studyLocusId"], cs["variantId"])), edges)
print("clusters:", len(components))
print("credible sets assigned:", sum(len(m) for _, m in components))

qualifying disease credible sets: 70618


colocalisation edges within the set: 843608


clusters: 20041
credible sets assigned: 70618


## Per-cluster counts

In [3]:
table = clusters.cluster_table(cs, components)
table.to_parquet(paper.derived("variant_clusters"), index=False)

print("clusters with more than one lead variant:", int((table["uniqueLeadVariants"] > 1).sum()))
print("clusters with more than one disease:", int((table["uniqueDiseases"] > 1).sum()))
print("clusters with more than one therapeutic area:", int((table["uniqueTherapeuticAreas"] > 1).sum()))
print(table[["uniqueDiseases", "uniqueTherapeuticAreas"]].agg(["min", "max", "mean"]).round(4).to_string())

clusters with more than one lead variant: 5595
clusters with more than one disease: 6617
clusters with more than one therapeutic area: 4539
      uniqueDiseases  uniqueTherapeuticAreas
min           1.0000                  1.0000
max         120.0000                 20.0000
mean          2.1415                  1.4046


## Cluster membership, for Supplementary Table 15

In [4]:
membership = clusters.membership_table(cs, components)
membership.to_parquet(paper.derived("cluster_membership"), index=False)
print("cluster-disease rows:", len(membership))
membership.head(3)

cluster-disease rows: 42918


,cluster_id,leadVariants,diseaseId,diseaseName,therapeuticArea,vPS
0,0,16_89579029_G_T,EFO_0000756,melanoma,cancer or benign tumor,2
1,0,16_89579029_G_T,EFO_0004279,suntan,other,2
2,1,2_66523432_G_T,EFO_0004270,restless legs syndrome,nervous system disease,5


## Control: the raw curator column reproduces the originally published counts

In [5]:
raw = clusters.cluster_table(cs, components, trait_column="traitFromSourceMappedIds")
comparison = pd.DataFrame(
    {
        "raw (originally published)": [
            (raw["uniqueDiseases"] > 1).sum(),
            raw["uniqueDiseases"].max(),
            raw["uniqueDiseases"].mean(),
            (raw["uniqueTherapeuticAreas"] > 1).sum(),
            raw["uniqueTherapeuticAreas"].max(),
            raw["uniqueTherapeuticAreas"].mean(),
        ],
        "resolved (used here)": [
            (table["uniqueDiseases"] > 1).sum(),
            table["uniqueDiseases"].max(),
            table["uniqueDiseases"].mean(),
            (table["uniqueTherapeuticAreas"] > 1).sum(),
            table["uniqueTherapeuticAreas"].max(),
            table["uniqueTherapeuticAreas"].mean(),
        ],
    },
    index=[
        "clusters with >1 disease",
        "max diseases",
        "mean diseases",
        "clusters with >1 therapeutic area",
        "max therapeutic areas",
        "mean therapeutic areas",
    ],
).round(4)
comparison

,raw (originally published),resolved (used here)
clusters with >1 disease,6678.0000,6617.0000
max diseases,122.0000,120.0000
mean diseases,2.1561,2.1415
clusters with >1 therapeutic area,4536.0000,4539.0000
max therapeutic areas,20.0000,20.0000
mean therapeutic areas,1.4025,1.4046
